In [4]:
%pip install torch transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 44.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 37.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 65.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 66.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 82.9 MB/s eta 0:00:00:00:01:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 44.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 17.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 141.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 69.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 184.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.8/807.8 kB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 148.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys

sys.path.append(
    "/home/sagemaker-user/adaptive-llm-routing"
)

import torch

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)

from app.router.baseline import BaselineRouter


# ============================================================
# CREATE V1 ROUTER
# ============================================================

router = BaselineRouter(
    max_label_samples=3000
)


# ============================================================
# LOAD DATA
# ============================================================

dataset = router.load_data()


# ============================================================
# PREPARE DATA
# ============================================================

queries, labels = router.prepare_data(
    dataset
)


# ============================================================
# TRAIN V1
# ============================================================

results = router.fit(
    queries,
    labels
)


# ============================================================
# TEST QUERIES
# ============================================================

test_queries = [
    "What is Python?",
    "What is the capital of France?",
    "Explain how neural networks work.",
    "Summarize this article.",
    "Compare RAG with fine-tuning.",
    "Debug this Python program.",
    "Design a scalable multi-agent AI system.",
    "Design a production RAG system with monitoring and fault tolerance.",
    "Prove why gradient descent converges under these assumptions.",
    "Give me three names for a coffee shop.",
    "Write a Python function that reverses a string.",
    "Explain the difference between TCP and UDP.",
    "Build an end-to-end machine learning pipeline.",
]


print("\n")
print("=" * 70)
print("V1 ROUTER TEST RESULTS")
print("=" * 70)


for i, query in enumerate(
    test_queries,
    start=1,
):

    result = router.route_model(
        query
    )

    print(
        f"\n[{i}] {query}"
    )

    print(
        f"    Complexity : "
        f"{result['complexity']}"
    )

    print(
        f"    Confidence : "
        f"{result['confidence']:.2%}"
    )

    print(
        f"    Model      : "
        f"{result['model']}"
    )

    print(
        f"    Cached     : "
        f"{result['cached']}"
    )


# ============================================================
# CACHE TEST
# ============================================================

print("\n")
print("=" * 70)
print("CACHE TEST")
print("=" * 70)


cache_query = (
    "Explain how gradient descent works."
)

first_result = router.route_model(
    cache_query
)

second_result = router.route_model(
    cache_query
)

print("\nFirst request:")
print(first_result)

print("\nSecond request:")
print(second_result)

print(
    "\nCache working:",
    second_result["cached"]
)


# ============================================================
# CUSTOM QUERY
# ============================================================

print("\n")
print("=" * 70)
print("CUSTOM QUERY TEST")
print("=" * 70)


custom_query = input(
    "\nEnter your own query: "
)

custom_result = router.route_model(
    custom_query
)

print("\nRouting result:")

print(
    f"Complexity : "
    f"{custom_result['complexity']}"
)

print(
    f"Confidence : "
    f"{custom_result['confidence']:.2%}"
)

print(
    f"Model      : "
    f"{custom_result['model']}"
)

print(
    f"Cached     : "
    f"{custom_result['cached']}"
)

ENVIRONMENT
Python: /home/sagemaker-user/.venv/bin/python
PyTorch: 2.14.1+cpu


/home/sagemaker-user/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Dolly dataset...
Dataset size: 15011
Columns: ['instruction', 'context', 'response', 'category']

Generating complexity labels for 3000 examples...

Loading zero-shot complexity labeler...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 9218.64it/s]


Labeled 100/3000
Labeled 200/3000
Labeled 300/3000
Labeled 400/3000
Labeled 500/3000
Labeled 600/3000
Labeled 700/3000
Labeled 800/3000
Labeled 900/3000
Labeled 1000/3000
Labeled 1100/3000
Labeled 1200/3000
Labeled 1300/3000
Labeled 1400/3000
Labeled 1500/3000
Labeled 1600/3000
Labeled 1700/3000
Labeled 1800/3000
Labeled 1900/3000
Labeled 2000/3000
Labeled 2100/3000
Labeled 2200/3000
Labeled 2300/3000
Labeled 2400/3000
Labeled 2500/3000
Labeled 2600/3000
Labeled 2700/3000
Labeled 2800/3000
Labeled 2900/3000
Labeled 3000/3000

Raw weak-label distribution:
  simple  : 213
  medium  : 452
  complex : 2335

Balanced weak-label distribution:
  simple  : 213
  medium  : 213
  complex : 213

Saved V1 labels to:
data/evaluation/v1_complexity_labels.json

Prepared examples: 639

Complexity distribution:
  simple  : 213
  medium  : 213
  complex : 213

Training V1...
Training examples: 511
Validation examples: 128

Fitting TF-IDF...
TF-IDF shape: (511, 913)

Training Logistic Regression...

V1 V

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/home/sagemaker-user/adaptive-llm-routing")

INPUT_FILE = (
    PROJECT_ROOT /
    "notebooks/data/evaluation/label_audit/label_review.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "notebooks/data/evaluation/label_audit"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

review = pd.read_csv(INPUT_FILE)

# ------------------------------------------------------------
# 1. Lowest-confidence examples
# ------------------------------------------------------------

low_conf = (
    review
    .sort_values("confidence", ascending=True)
    .head(100)
    .copy()
)

# ------------------------------------------------------------
# 2. Random control sample from high-confidence examples
# ------------------------------------------------------------

high_conf_pool = review[
    ~review["instruction"].isin(low_conf["instruction"])
].copy()

high_conf = (
    high_conf_pool
    .sort_values("confidence", ascending=False)
    .head(25)
    .copy()
)

# ------------------------------------------------------------
# 3. Combine
# ------------------------------------------------------------

audit = pd.concat(
    [low_conf, high_conf],
    ignore_index=True
)

# Add columns for human review
audit["review_label"] = ""
audit["review_status"] = "pending"
audit["review_notes"] = ""

# Give each row a stable review ID
audit.insert(
    0,
    "review_id",
    range(1, len(audit) + 1)
)

# Keep only useful columns
audit = audit[
    [
        "review_id",
        "instruction",
        "complexity",
        "confidence",
        "review_label",
        "review_status",
        "review_notes"
    ]
]

OUTPUT_FILE = OUTPUT_DIR / "v1_label_review_125.csv"

audit.to_csv(
    OUTPUT_FILE,
    index=False
)

print("=" * 70)
print("LABEL REVIEW FILE CREATED")
print("=" * 70)

print(f"Total examples: {len(audit)}")
print(f"Low-confidence examples: {len(low_conf)}")
print(f"High-confidence control examples: {len(high_conf)}")

print(f"\nSaved to:")
print(OUTPUT_FILE)

print("\nReview instructions:")
print("review_label must be: simple / medium / complex")
print("review_status should be: corrected / confirmed")
print("review_notes is optional.")

display(audit.head(20))